# 00 — Kiểm tra thư mục dự án DSP

**Mục tiêu:** hiểu vị trí dữ liệu, các thư mục và biểu mẫu trước khi bắt đầu DSP.

**Đầu vào:** thư mục dự án đã thiết lập. **Đầu ra:** thông báo ngay dưới từng ô code; không ghi vào raw.

Notebook này kiểm tra cấu trúc và header CSV, chưa audit nhãn, subject ID, split hoặc hiệu quả thuật toán. Không đọc nội dung audio.

**Cách học:** đọc phần giải thích → chạy ô code ngay dưới bằng `Shift + Enter` → xem kết quả → sang bước tiếp theo. Trước khi xác nhận một buổi làm, dùng **Restart Kernel and Run All Cells** để kiểm tra notebook chạy từ trạng thái sạch. Nếu chạy bằng JupyterLab, mở từ thư mục dự án hoặc thư mục con của dự án.

## 1. Xem kernel và tìm thư mục dự án

**Vị trí file notebook và thư mục làm việc của kernel có thể khác nhau.** Mở file trong `dsp/notebooks` chưa bảo đảm `Path.cwd()` đang ở thư mục đó.

Chạy ô dưới để xem thư mục hiện tại, hệ điều hành và Python thực tế trước khi kiểm tra đường dẫn.

- Để `PROJECT_PATH = ""` nếu kernel đang ở trong thư mục dự án: code sẽ tìm ở thư mục hiện tại và các thư mục cha.
- Nếu kernel ở chỗ khác, điền đường dẫn gốc dự án (thư mục chứa `requirements-dsp.txt`, `README.md`, `dsp`), không điền đường dẫn tới file notebook.
- Ví dụ trên Windows: `PROJECT_PATH = r"C:\path\to\DSP-GAP"`. Đây là đường dẫn đầu vào, bạn có thể đổi theo nơi lưu dự án.
- Nếu kernel báo Linux, đường dẫn `D:\...` không dùng được trực tiếp. Chọn Python Windows trong VS Code/Jupyter, hoặc dùng đường dẫn mount thật của môi trường Linux. Với WSL, ổ D thường được mount ở `/mnt/d/`; chỉ dùng khi thư mục thực sự tồn tại.

Không tạo thư mục giả để vượt lỗi. Code chỉ đi tiếp khi tìm được dự án thật. `errors` được đặt lại về 0 ở đầu mỗi lần chạy sạch.

In [1]:
import csv
from pathlib import Path, PureWindowsPath
import sys

# Đầu vào tùy chọn: để trống để tự tìm, hoặc điền thư mục gốc dự án.
PROJECT_PATH = ""

current_dir = Path.cwd().resolve()
print("Thư mục kernel:", current_dir)
print("Hệ điều hành kernel:", sys.platform)
print("Python kernel:", sys.executable)
print("Phiên bản Python:", sys.version.split()[0])

if PROJECT_PATH:
    if sys.platform != "win32" and PureWindowsPath(PROJECT_PATH).drive:
        raise ValueError(
            "Kernel đang chạy Linux/macOS nên không đọc trực tiếp đường dẫn ổ D: của Windows. "
            "Chọn Python Windows trên máy, hoặc điền đường dẫn mount thật trong môi trường này."
        )
    candidates = [Path(PROJECT_PATH).expanduser().resolve()]
else:
    candidates = [current_dir, *current_dir.parents]

PROJECT_ROOT = None
for folder in candidates:
    if (folder / "requirements-dsp.txt").is_file() and (folder / "dsp/notebooks").is_dir():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy dự án từ thư mục kernel {current_dir}. "
        "Điền PROJECT_PATH bằng đường dẫn thư mục gốc có requirements-dsp.txt và dsp/notebooks, "
        "hoặc mở Jupyter từ thư mục dự án. Nếu kernel là Linux, phải kiểm tra đường dẫn mount/kernel."
    )

errors = 0
print("Thư mục dự án:", PROJECT_ROOT)

Thư mục kernel: [project]\dsp\notebooks
Hệ điều hành kernel: win32
Python kernel: [project]\.venv\Scripts\python.exe
Phiên bản Python: 3.11.9
Thư mục dự án: [project]


## 2. Kiểm tra tài liệu

Danh sách dưới đây là các file cần có. `is_file()` chỉ kiểm tra sự tồn tại, không thực thi hay sửa nội dung file. `[MISSING]` có nghĩa là cần xem lại vị trí hoặc tên file.

In [2]:
REQUIRED_FILES = [
    "README.md",
    "requirements-dsp.txt",
    "dsp/scripts/download_biocas2022.py",
    "dsp/notebooks/01_kiem_ke_du_lieu.ipynb",
    "dsp/notebooks/02_chia_tap_theo_nguoi.ipynb",
    "dsp/notebooks/03_tao_segment_va_kiem_tra.ipynb",
    "dsp/notebooks/04_baseline_real_only.ipynb",
]
for relative_path in REQUIRED_FILES:
    exists = (PROJECT_ROOT / relative_path).is_file()
    print(f"[{'OK' if exists else 'MISSING'}] {relative_path}")
    if not exists:
        errors += 1


[OK] README.md
[OK] requirements-dsp.txt
[OK] dsp/scripts/download_biocas2022.py
[OK] dsp/notebooks/01_kiem_ke_du_lieu.ipynb
[OK] dsp/notebooks/02_chia_tap_theo_nguoi.ipynb
[OK] dsp/notebooks/03_tao_segment_va_kiem_tra.ipynb
[OK] dsp/notebooks/04_baseline_real_only.ipynb


## 3. Kiểm tra thư mục

`raw`: dữ liệu gốc; `manifests`: danh sách nguồn/split; `processed`: dữ liệu dẫn xuất; `synthetic`: âm thanh tổng hợp; `outputs`: báo cáo, hình và kết quả chạy.

Thư mục tồn tại không có nghĩa là bước nghiên cứu đó đã hoàn thành.

In [3]:
REQUIRED_DIRS = [
    "dsp/data/raw",
    "dsp/data/manifests",
    "dsp/data/processed",
    "dsp/data/synthetic",
    "dsp/outputs/reports",
    "dsp/outputs/figures",
    "dsp/outputs/runs",
]

for relative_path in REQUIRED_DIRS:
    exists = (PROJECT_ROOT / relative_path).is_dir()
    print(f"[{'OK' if exists else 'MISSING'}] {relative_path}/")
    if not exists:
        errors += 1

[OK] dsp/data/raw/
[OK] dsp/data/manifests/
[OK] dsp/data/processed/
[OK] dsp/data/synthetic/
[OK] dsp/outputs/reports/
[OK] dsp/outputs/figures/
[OK] dsp/outputs/runs/


## 4. Đọc tên cột của các biểu mẫu

Các CSV trong `templates` chỉ định nghĩa tên cột, chưa chứa dữ liệu thật. Ví dụ `anchor_event_id` sẽ nối một mẫu tổng hợp với sự kiện nguồn. Giải thích từng cột nằm trong `dsp/data/README.md`.

Đây là danh sách cần đối chiếu. Chưa cần chỉnh; khi đổi schema, phải đổi template, tài liệu và phần kiểm tra cùng nhau.

In [4]:
TEMPLATE_HEADERS = {
    "events.csv": "event_id,recording_id,subject_id,audio_path,annotation_path,start_ms,end_ms,source_label,label,official_partition,split,included,exclusion_reason",
    "subjects.csv": "subject_id,official_partition,split,split_seed,notes",
    "calibration.csv": "parameter,unit,value,evidence_path,status,frozen_on,notes",
    "synthetic_provenance.csv": "candidate_id,repetition,seed,anchor_event_id,anchor_subject_id,background_event_id,background_subject_id,attempt_index,controls_path,accepted,rejection_reason,audio_path,target_duration_ms,achieved_duration_ms,target_f50_hz,achieved_f50_hz",
    "draws.csv": "repetition,seed,policy,draw_index,candidate_id,anchor_event_id",
    "run_log.csv": "run_id,created_at,stage,status,command,seed,input_manifest,parameters_path,environment_path,output_dir,notes",
}

## 5. Kiểm tra header CSV

Mở từng template bằng UTF-8, đọc dòng đầu và so sánh với danh sách ở bước 4. Nếu file thiếu hoặc header khác thì ghi lỗi, tiếp tục kiểm tra những file còn lại.

In [5]:
for name, expected in TEMPLATE_HEADERS.items():
    path = PROJECT_ROOT / "dsp/templates" / name
    try:
        with path.open(encoding="utf-8-sig", newline="") as file:
            header = next(csv.reader(file), [])
        if header != expected.split(","):
            print(f"[ERROR] Header không khớp: {name}")
            errors += 1
        else:
            print(f"[OK] {name}: {len(header)} cột")
    except (OSError, UnicodeError, csv.Error) as error:
        print(f"[ERROR] {name}: {error}")
        errors += 1

[OK] events.csv: 13 cột
[OK] subjects.csv: 5 cột
[OK] calibration.csv: 7 cột
[OK] synthetic_provenance.csv: 16 cột
[OK] draws.csv: 6 cột
[OK] run_log.csv: 11 cột


## 6. Xác định vị trí BioCAS2022

Chỉ kiểm tra tên bốn thư mục cấp trên, không đọc audio hay nội dung annotation test. Kết quả tải và checksum đã lưu trong báo cáo download. Audit nội dung sẽ là notebook tiếp theo.

In [6]:
DATASET_ROOT = PROJECT_ROOT / "dsp/data/raw/BioCAS2022"
print("Dataset:", DATASET_ROOT)
for name in ["train2022_wav", "train2022_json", "test2022_wav", "test2022_json"]:
    exists = (DATASET_ROOT / name).is_dir()
    print(f"[{'OK' if exists else 'INFO: chưa tải'}] {name}/")
print("Nếu chưa có dataset: python dsp/scripts/download_biocas2022.py")


Dataset: [project]\dsp\data\raw\BioCAS2022
[OK] train2022_wav/
[OK] train2022_json/
[OK] test2022_wav/
[OK] test2022_json/
Nếu chưa có dataset: python dsp/scripts/download_biocas2022.py


## 7. Kết luận kiểm tra

`PASS` chỉ xác nhận các mục vừa kiểm tra. Nó chưa xác nhận subject independence hoặc chất lượng synthesis.

Nếu có lỗi, xem ô phía trên, sửa đường dẫn/file tương ứng rồi **restart kernel và chạy toàn bộ từ đầu** để tránh giữ biến cũ. Chạy lặp một ô có `errors += 1` có thể cộng lỗi nhiều lần trong cùng kernel; kết quả xác nhận luôn lấy từ lần chạy sạch.

In [7]:
if errors:
    print(f"FAIL: còn {errors} lỗi cấu trúc/header.")
else:
    print("PASS: cấu trúc source và template; dataset được thông báo riêng ở trên.")

print("Chưa audit nhãn/subject/split. Chưa chạy thuật toán DSP.")
assert errors == 0, "Xem các dòng MISSING/ERROR, sửa rồi Restart Kernel and Run All Cells."

PASS: cấu trúc source và template; dataset được thông báo riêng ở trên.
Chưa audit nhãn/subject/split. Chưa chạy thuật toán DSP.


## Sau notebook này

Mở [01_kiem_ke_du_lieu.ipynb](01_kiem_ke_du_lieu.ipynb) để kiểm kê annotation, xác minh subject ID theo README và xem bảng số lượng/vấn đề trước chia tập. Notebook 01 đã được tạo và chạy kiểm tra ngày 27/10/2026.

Mỗi notebook có mục tiêu, đầu vào, tham số, các ô xử lý, kết quả và hạn chế. Mỗi bước lưu artifact rõ ràng để bước sau đọc lại; không phụ thuộc biến còn nằm trong kernel của notebook trước.